# Baseline Experiments

Systematic evaluation of baseline approaches for jaguar re-identification:
- **Random Baseline**: Random embeddings (sanity check, should get ~0% mAP)
- **Backbone-Only Baselines**: Pre-trained backbone features without fine-tuning (14 backbones)

**Tested Backbones:**
- **DINOv3-Large** (vit_large_patch16_dinov3.lvd1689m, 1024-dim, latest self-supervised ViT)
- **DINOv3-Base** (vit_base_patch16_dinov3.lvd1689m, 768-dim, efficient high-quality)
- **MiewID-MSv2** (conservationxlabs/miewid-msv2, 2152-dim, wildlife re-ID specialist)
- **MiewID-MSv3** (conservationxlabs/miewid-msv3, 2152-dim, wildlife re-ID specialist)
- **MegaDescriptor-L-384** (1536-dim, our main model, trained on wildlife datasets)
- **MegaDescriptor-B-224** (768-dim, animal re-ID specialist)
- DINOv2-Large (1024-dim, previous-generation self-supervised)
- DINOv2-Base (768-dim)
- DINOv2-Small (384-dim)
- ResNet50 (2048-dim)
- ConvNeXt Base (1024-dim)
- ConvNeXtV2 Base (1024-dim)
- EfficientNet B3 (1536-dim)
- **EfficientNetV2-RW-M** (hf-hub:timm/efficientnetv2_rw_m.agc_in1k, 2152-dim)

**Dataset Source**: Hugging Face dataset `JaguarCameraTrap/jaguars_camera_trap_0226-segmented_deduplicated`.

Results logged to Wandb project: `camera-trap-reidentification`, group: `baselines`

## Setup

In [ ]:
import sys
from pathlib import Path
import logging
import importlib

# Add src to path
project_root = Path.cwd().parent / "camera-trap-footage"
sys.path.insert(0, str(project_root / "src"))

from jaguars.common.logging_utils import setup_logger
from jaguars.reidentification.config import get_default_config
from jaguars.reidentification.experiments import get_baseline_experiments
from jaguars.reidentification.evaluation.evaluation import run_processing as run_evaluation
from jaguars.reidentification.wandb_results import fetch_latest_metrics_for_experiments

# Force reload to get the latest code
import jaguars.reidentification.experiments
importlib.reload(jaguars.reidentification.experiments)
from jaguars.reidentification.experiments import get_baseline_experiments

logger = setup_logger("baseline_experiments", level=logging.INFO)
print("✓ Imports successful")

✓ Imports successful


## Configuration

Configure base settings for all baseline experiments.

In [10]:
# Get default configuration
config = get_default_config()

# Run metadata for clear WandB separation
RUN_BATCH = "hf_0226_segmented_deduplicated_v2_cached"
DATASET_TAG = "dataset:fiftyone_JID_HF_0226_Segmented_Deduplicated_Cached"
SOURCE_TAG = "source:fiftyone_local_cache"

# Prepared local dataset
LOCAL_FO_DATASET_NAME = "JID_HF_0226_Segmented_Deduplicated_Cached"

# Wandb settings
config.wandb.enabled = True
config.wandb.entity = "jaguars"
config.wandb.project = "camera-trap-reidentification"
config.wandb.tags = ["baselines", DATASET_TAG, SOURCE_TAG, f"run_batch:{RUN_BATCH}"]

# Dataset settings (prepared local FiftyOne cache)
config.dataset.source = "fiftyone"
config.dataset.fo_dataset_name = LOCAL_FO_DATASET_NAME
config.dataset.fo_split_field = "closed_set_split"
config.dataset.fo_patches_field = "sam3_segmentations"
config.dataset.fo_label_field = "ground_truth"
config.dataset.fo_embeddings_field = None  # backbone-specific cached field auto-detection

# Keep standard split names
config.dataset.train_split = "train"
config.dataset.val_split = "val"
config.dataset.test_split = "test"

# Training settings
config.training.num_epochs = 0  # Baselines don't train

print(f"✓ Base config loaded")
print(f"  Wandb project: {config.wandb.project}")
print(f"  Wandb tags: {config.wandb.tags}")
print(f"  Dataset source: {config.dataset.source}")
print(f"  FiftyOne dataset: {config.dataset.fo_dataset_name}")
print(f"  Split field: {config.dataset.fo_split_field}")
print(f"  Label field: {config.dataset.fo_label_field}")
print(f"  Patches field: {config.dataset.fo_patches_field}")

✓ Base config loaded
  Wandb project: camera-trap-reidentification
  Wandb tags: ['baselines', 'dataset:fiftyone_JID_HF_0226_Segmented_Deduplicated_Cached', 'source:fiftyone_local_cache', 'run_batch:hf_0226_segmented_deduplicated_v2_cached']
  Dataset source: fiftyone
  FiftyOne dataset: JID_HF_0226_Segmented_Deduplicated_Cached
  Split field: closed_set_split
  Label field: ground_truth
  Patches field: sam3_segmentations


In [11]:
import fiftyone as fo

if not fo.dataset_exists(LOCAL_FO_DATASET_NAME):
    raise ValueError(
        f"Local FiftyOne dataset '{LOCAL_FO_DATASET_NAME}' not found. "
        "Run notebooks/prepare_fiftyone_cache.ipynb first."
    )

dataset = fo.load_dataset(LOCAL_FO_DATASET_NAME)
images_view = dataset.select_group_slices("image") if dataset.group_field else dataset
print(f"✓ Using prepared local FiftyOne dataset: {LOCAL_FO_DATASET_NAME}")
print(f"  Total samples: {len(dataset)}")
print(f"  Image samples: {len(images_view)}")
print(f"  Group field: {dataset.group_field}")
print(f"  Splits: {images_view.count_values(config.dataset.fo_split_field)}")

✓ Using prepared local FiftyOne dataset: JID_HF_0226_Segmented_Deduplicated_Cached
  Total samples: 1998
  Image samples: 1998
  Group field: group
  Splits: {'train': 1632, 'val': 167, 'test': 199}


## Get Baseline Experiments

In [12]:
# Get baseline experiments with our custom config
baseline_experiments = get_baseline_experiments(base_config=config)

# Ensure all runs carry the same dataset/run-batch tags and a unique prefix
for exp in baseline_experiments:
    exp.base_config.wandb.tags = list(dict.fromkeys(exp.base_config.wandb.tags + [DATASET_TAG, SOURCE_TAG, f"run_batch:{RUN_BATCH}"]))
    run_base = exp.base_config.wandb.run_name or exp.name
    exp.base_config.wandb.run_name = f"{RUN_BATCH}__{run_base}"

print(f"✓ {len(baseline_experiments)} baseline experiments configured:")
for exp in baseline_experiments:
    print(f"  - {exp.name}: {exp.description}")

✓ 15 baseline experiments configured:
  - baseline_random: Random embeddings (sanity check - should be near 0% mAP)
  - baseline_conservationxlabs_miewid-msv2: MiewID-MSv2 without fine-tuning
  - baseline_conservationxlabs_miewid-msv3: MiewID-MSv3 without fine-tuning
  - baseline_hf-hub:BVRA_MegaDescriptor-L-384: BVRA MegaDescriptor Large 384 without fine-tuning
  - baseline_hf-hub:BVRA_MegaDescriptor-B-224: BVRA MegaDescriptor Base 224 without fine-tuning
  - baseline_vit_large_patch14_dinov2.lvd142m: DINOv2 Large without fine-tuning
  - baseline_vit_base_patch14_dinov2.lvd142m: DINOv2 Base without fine-tuning
  - baseline_vit_small_patch14_dinov2.lvd142m: DINOv2 Small without fine-tuning
  - baseline_resnet50: ResNet50 without fine-tuning
  - baseline_convnext_base: ConvNeXt Base without fine-tuning
  - baseline_convnextv2_base.fcmae_ft_in22k_in1k: ConvNeXtV2 Base without fine-tuning
  - baseline_efficientnet_b3: EfficientNet B3 without fine-tuning
  - baseline_hf-hub:timm_efficientn

## Run Experiments

Execute all baseline experiments and log to Wandb.

In [13]:
# Run all baseline experiments
results = {}

for experiment in baseline_experiments:
    logger.info(f"Running experiment: {experiment.name}")
    logger.info(f"  Description: {experiment.description}")
    logger.info(f"  Group: {experiment.group}")
    
    try:
        # Determine baseline mode based on loss name
        baseline_mode = experiment.base_config.baseline_mode
        
        # Run evaluation (for baselines, this skips model loading)
        result = run_evaluation(
            config=experiment.base_config,
            baseline_mode=baseline_mode,
            verbose=True
        )
        results[experiment.name] = result
        logger.info(f"✓ {experiment.name} completed")
    except Exception as e:
        logger.error(f"✗ {experiment.name} failed: {e}")
        import traceback
        traceback.print_exc()
        results[experiment.name] = {"error": str(e)}

print(f"\n✓ All {len(baseline_experiments)} baseline experiments completed")

11:11:57 - jid_logger.baseline_experiments - INFO - Running experiment: baseline_random
11:11:57 - jid_logger.baseline_experiments - INFO -   Description: Random embeddings (sanity check - should be near 0% mAP)
11:11:57 - jid_logger.baseline_experiments - INFO -   Group: baselines
11:11:57 - jid_logger.reidentification.evaluation - INFO - Starting evaluation...
11:11:57 - jid_logger.reidentification.evaluation - INFO - Dataset source: fiftyone
11:11:57 - jid_logger.reidentification.evaluation - INFO - Resource validation passed


11:11:59 - jid_logger.reidentification.evaluation - INFO - Loading dataset...
11:12:29 - jid_logger.reidentification.evaluation - INFO - Train set: 1632 samples, 170 classes
11:12:29 - jid_logger.reidentification.evaluation - INFO - Test set: 199 samples, 74 classes
11:12:29 - jid_logger.reidentification.evaluation - INFO - Running random baseline evaluation (no training)
11:12:29 - jid_logger.reidentification.evaluation - INFO - Computing comprehensive metrics...
11:12:29 - jid_logger.reidentification.evaluation - INFO - Sample-level mAP: 0.0483
11:12:29 - jid_logger.reidentification.evaluation - INFO - Identity-balanced mAP: 0.0409
11:12:29 - jid_logger.reidentification.evaluation - INFO - Closed-set mAP (≥3 train & test): 0.0405 (27/41 identities)
11:12:29 - jid_logger.reidentification.evaluation - INFO - CMC@1: 0.0120
11:12:29 - jid_logger.reidentification.evaluation - INFO - CMC@5: 0.1386
11:12:29 - jid_logger.reidentification.evaluation - INFO - Saved embeddings to data/results/r

closed_set_map,▁
closed_set_num_identities,▁
cmc@1,▁
cmc@10,▁
cmc@20,▁
cmc@5,▁
identity_balanced_map,▁
map,▁
map_train_10+_val_0-2,▁
map_train_10+_val_10+,▁
+14,...


11:12:30 - jid_logger.reidentification.evaluation - INFO - Evaluation completed!
11:12:30 - jid_logger.baseline_experiments - INFO - ✓ baseline_random completed
11:12:30 - jid_logger.baseline_experiments - INFO - Running experiment: baseline_conservationxlabs_miewid-msv2
11:12:30 - jid_logger.baseline_experiments - INFO -   Description: MiewID-MSv2 without fine-tuning
11:12:30 - jid_logger.baseline_experiments - INFO -   Group: baselines
11:12:30 - jid_logger.reidentification.evaluation - INFO - Starting evaluation...
11:12:30 - jid_logger.reidentification.evaluation - INFO - Dataset source: fiftyone
11:12:30 - jid_logger.reidentification.evaluation - INFO - Resource validation passed


11:12:32 - jid_logger.reidentification.evaluation - INFO - Loading dataset...
11:13:03 - jid_logger.reidentification.evaluation - INFO - Train set: 1632 samples, 170 classes
11:13:03 - jid_logger.reidentification.evaluation - INFO - Test set: 199 samples, 74 classes
11:13:03 - jid_logger.reidentification.evaluation - INFO - Running backbone-only evaluation (no fine-tuning)
11:13:03 - jid_logger.reidentification.evaluation - INFO - Extracting test embeddings with backbone...
Loading conservationxlabs/miewid-msv2 model via transformers AutoModel...
Building Model Backbone for efficientnetv2_rw_m model
config.model_name efficientnetv2_rw_m
model_name efficientnetv2_rw_m
final_in_features 2152


...

11:41:57 - jid_logger.reidentification.evaluation - INFO - Extracting train embeddings with backbone...



Train embeddings: 100%|██████████| 51/51 [01:50<00:00,  2.16s/it]

11:43:48 - jid_logger.reidentification.evaluation - INFO - Computing comprehensive metrics...
11:43:48 - jid_logger.reidentification.evaluation - INFO - Sample-level mAP: 0.3475
11:43:48 - jid_logger.reidentification.evaluation - INFO - Identity-balanced mAP: 0.3445
11:43:48 - jid_logger.reidentification.evaluation - INFO - Closed-set mAP (≥3 train & test): 0.3699 (27/41 identities)
11:43:48 - jid_logger.reidentification.evaluation - INFO - CMC@1: 0.5060
11:43:48 - jid_logger.reidentification.evaluation - INFO - CMC@5: 0.6506
11:43:48 - jid_logger.reidentification.evaluation - INFO - Saved embeddings to data/results/reidentification/test_embeddings.npy
11:43:48 - jid_logger.reidentification.evaluation - INFO - Saved predictions to data/results/reidentification/test_predictions.json


closed_set_map,▁
closed_set_num_identities,▁
cmc@1,▁
cmc@10,▁
cmc@20,▁
cmc@5,▁
identity_balanced_map,▁
map,▁
map_train_10+_val_0-2,▁
map_train_10+_val_10+,▁
+14,...


11:43:49 - jid_logger.reidentification.evaluation - INFO - Evaluation completed!
11:43:49 - jid_logger.baseline_experiments - INFO - ✓ baseline_vit_base_patch16_dinov3.lvd1689m completed

✓ All 15 baseline experiments completed


## Summary

Display results from all baseline experiments.

In [14]:
results

{'baseline_random': {'status': 'completed',
  'num_test_samples': 199,
  'num_classes': 74,
  'map': 0.04826187153583472,
  'identity_balanced_map': 0.04090800657189146,
  'cmc_curve': [0.012048192771084338,
   0.04216867469879518,
   0.07228915662650602,
   0.0963855421686747,
   0.13855421686746988,
   0.15060240963855423,
   0.1686746987951807,
   0.18674698795180722,
   0.19879518072289157,
   0.22289156626506024,
   0.24096385542168675,
   0.26506024096385544,
   0.27710843373493976,
   0.3072289156626506,
   0.3132530120481928,
   0.3132530120481928,
   0.3373493975903614,
   0.3373493975903614,
   0.3493975903614458,
   0.3614457831325301,
   0.3614457831325301,
   0.37349397590361444,
   0.39156626506024095,
   0.39759036144578314,
   0.40963855421686746,
   0.42168674698795183,
   0.42771084337349397,
   0.4397590361445783,
   0.4578313253012048,
   0.463855421686747,
   0.4819277108433735,
   0.5,
   0.5180722891566265,
   0.536144578313253,
   0.5481927710843374,
   0.560240

In [ ]:
# Print summary of latest W&B results
import pandas as pd

MODEL_NAME_MAP = {
    "vit_large_patch16_dinov3.lvd1689m": "DINOv3-Large",
    "vit_base_patch16_dinov3.lvd1689m": "DINOv3-Base",
    "conservationxlabs/miewid-msv2": "MiewID-MSv2",
    "conservationxlabs/miewid-msv3": "MiewID-MSv3",
    "hf-hub:BVRA/MegaDescriptor-L-384": "MegaDescriptor-L-384",
    "hf-hub:BVRA/MegaDescriptor-B-224": "MegaDescriptor-B-224",
    "vit_large_patch14_dinov2.lvd142m": "DINOv2-Large",
    "vit_base_patch14_dinov2.lvd142m": "DINOv2-Base",
    "vit_small_patch14_dinov2.lvd142m": "DINOv2-Small",
    "resnet50": "ResNet50",
    "convnext_base": "ConvNeXt-Base",
    "convnextv2_base.fcmae_ft_in22k_in1k": "ConvNeXtV2-Base",
    "efficientnet_b3": "EfficientNet-B3",
    "hf-hub:timm/efficientnetv2_rw_m.agc_in1k": "EfficientNetV2-RW-M",
}

baseline_lookup = {exp.name: exp for exp in baseline_experiments}

def _display_name(exp_name: str) -> str:
    if exp_name == "random_baseline":
        return "Random baseline"
    exp = baseline_lookup.get(exp_name)
    if exp is None:
        return exp_name.replace("_", " ")
    backbone_name = exp.base_config.backbone.name
    return MODEL_NAME_MAP.get(backbone_name, backbone_name)

wandb_results = fetch_latest_metrics_for_experiments(
    experiments=baseline_experiments,
    entity=config.wandb.entity,
    project=config.wandb.project,
    additional_tags=[DATASET_TAG, SOURCE_TAG, f"run_batch:{RUN_BATCH}"],
)

summary_data = []
for exp in baseline_experiments:
    exp_name = exp.name
    result = wandb_results.get(exp_name, {"error": "Missing W&B result"})
    display_name = _display_name(exp_name)
    if "error" in result:
        summary_data.append({
            "Experiment": display_name,
            "mAP": "ERROR",
            "CMC@1": "ERROR",
            "mAP (>=9 total)": "ERROR",
        })
    else:
        map_val = result.get("map", "N/A")
        cmc1 = result.get("cmc@1", "N/A")
        map_9 = result.get("map_min_total_9", "N/A")

        map_str = f"{map_val:.4f}" if isinstance(map_val, (int, float)) else str(map_val)
        cmc1_str = f"{cmc1:.4f}" if isinstance(cmc1, (int, float)) else str(cmc1)
        map_9_str = f"{map_9:.4f}" if isinstance(map_9, (int, float)) else str(map_9)

        summary_data.append({
            "Experiment": display_name,
            "mAP": map_str,
            "CMC@1": cmc1_str,
            "mAP (>=9 total)": map_9_str,
        })

summary_df = pd.DataFrame(summary_data)
print("\n=== Baseline Results Summary (Latest W&B Runs) ===")
print(summary_df.to_string(index=False))
print(f"\nView detailed results at: https://wandb.ai/{config.wandb.entity}/{config.wandb.project}")


=== Baseline Results Summary ===
          Experiment    mAP  CMC@1 mAP (>=9 total)
        DINOv2-Large 0.0483 0.0120             N/A
         MiewID-MSv2 0.3042 0.4578             N/A
         MiewID-MSv3 0.3204 0.4518             N/A
MegaDescriptor-L-384 0.1270 0.1446             N/A
MegaDescriptor-B-224 0.2951 0.4639             N/A
        DINOv2-Large 0.2057 0.3373             N/A
         DINOv2-Base 0.2316 0.3795             N/A
        DINOv2-Small 0.2325 0.3434             N/A
            ResNet50 0.2673 0.3976             N/A
       ConvNeXt-Base 0.2531 0.3916             N/A
     ConvNeXtV2-Base 0.2439 0.4036             N/A
     EfficientNet-B3 0.2455 0.3675             N/A
 EfficientNetV2-RW-M 0.2486 0.3855             N/A
        DINOv3-Large 0.3334 0.4940             N/A
         DINOv3-Base 0.3475 0.5060             N/A

View detailed results at: https://wandb.ai/jaguars/camera-trap-reidentification


## Export for Report

Generate report-ready artifacts (CSV, LaTeX table, PNG figure).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

MODEL_NAME_MAP = {
    "vit_large_patch16_dinov3.lvd1689m": "DINOv3-Large",
    "vit_base_patch16_dinov3.lvd1689m": "DINOv3-Base",
    "conservationxlabs/miewid-msv2": "MiewID-MSv2",
    "conservationxlabs/miewid-msv3": "MiewID-MSv3",
    "hf-hub:BVRA/MegaDescriptor-L-384": "MegaDescriptor-L-384",
    "hf-hub:BVRA/MegaDescriptor-B-224": "MegaDescriptor-B-224",
    "vit_large_patch14_dinov2.lvd142m": "DINOv2-Large",
    "vit_base_patch14_dinov2.lvd142m": "DINOv2-Base",
    "vit_small_patch14_dinov2.lvd142m": "DINOv2-Small",
    "resnet50": "ResNet50",
    "convnext_base": "ConvNeXt-Base",
    "convnextv2_base.fcmae_ft_in22k_in1k": "ConvNeXtV2-Base",
    "efficientnet_b3": "EfficientNet-B3",
    "hf-hub:timm/efficientnetv2_rw_m.agc_in1k": "EfficientNetV2-RW-M",
}

baseline_lookup = {exp.name: exp for exp in baseline_experiments}

def _display_name(exp_name: str) -> str:
    if exp_name == "random_baseline":
        return "Random baseline"
    exp = baseline_lookup.get(exp_name)
    if exp is None:
        return exp_name.replace("_", " ")
    backbone_name = exp.base_config.backbone.name
    return MODEL_NAME_MAP.get(backbone_name, backbone_name)

# Build summary_df from latest W&B runs if this cell is run standalone
if "summary_df" not in globals():
    wandb_results = fetch_latest_metrics_for_experiments(
        experiments=baseline_experiments,
        entity=config.wandb.entity,
        project=config.wandb.project,
        additional_tags=[DATASET_TAG, SOURCE_TAG, f"run_batch:{RUN_BATCH}"],
    )

    summary_data = []
    for exp in baseline_experiments:
        exp_name = exp.name
        result = wandb_results.get(exp_name, {"error": "Missing W&B result"})
        display_name = _display_name(exp_name)
        if "error" in result:
            summary_data.append({
                "Experiment": display_name,
                "mAP": "ERROR",
                "CMC@1": "ERROR",
                "Closed-set mAP": "ERROR",
            })
        else:
            map_val = result.get("identity_balanced_map", result.get("map", "N/A"))
            cmc1 = result.get("cmc@1", "N/A")
            cs_map = result.get("closed_set_map", "N/A")
            summary_data.append({
                "Experiment": display_name,
                "mAP": f"{map_val:.4f}" if isinstance(map_val, (int, float)) else str(map_val),
                "CMC@1": f"{cmc1:.4f}" if isinstance(cmc1, (int, float)) else str(cmc1),
                "Closed-set mAP": f"{cs_map:.4f}" if isinstance(cs_map, (int, float)) else str(cs_map),
            })
    summary_df = pd.DataFrame(summary_data)

run_batch = globals().get("RUN_BATCH", "manual_run")
output_dir = Path("notebooks/data/results/figures/report") / run_batch / "baseline"
output_dir.mkdir(parents=True, exist_ok=True)

# Save tabular artifacts
csv_path = output_dir / "baseline_summary.csv"
tex_path = output_dir / "baseline_summary.tex"
summary_df.to_csv(csv_path, index=False)
summary_df.to_latex(tex_path, index=False)

# Save figure
plot_df = summary_df.copy()
plot_df["mAP_numeric"] = pd.to_numeric(plot_df["mAP"], errors="coerce")
plot_df = plot_df.dropna(subset=["mAP_numeric"]).sort_values("mAP_numeric", ascending=False)

fig_path = output_dir / "baseline_map_bar.png"
plt.figure(figsize=(12, max(4, 0.4 * len(plot_df))))
plt.barh(plot_df["Experiment"], plot_df["mAP_numeric"])
plt.gca().invert_yaxis()
plt.xlabel("Identity-balanced mAP")
plt.title("Baseline Comparison")
plt.tight_layout()
plt.savefig(fig_path, dpi=300)
plt.close()

print(f"Saved CSV: {csv_path}")
print(f"Saved LaTeX table: {tex_path}")
print(f"Saved figure: {fig_path}")

Saved CSV: notebooks/data/results/figures/report/hf_0226_segmented_deduplicated_v2_cached/baseline/baseline_summary.csv
Saved LaTeX table: notebooks/data/results/figures/report/hf_0226_segmented_deduplicated_v2_cached/baseline/baseline_summary.tex
Saved figure: notebooks/data/results/figures/report/hf_0226_segmented_deduplicated_v2_cached/baseline/baseline_map_bar.png
